# DZ14 — Логістична регресія: класифікація фруктів

У цьому завданні будуємо **класіфікатор**, який за рівнем солодкості фрукта (ознака `sweet`) визначає, чи це **соковитий манго (клас 1)**, чи **кислий лимон (клас 0)**.

Маємо датафрейм з двома стовпчиками:
- `sweet` — рівень солодкості;
- `let` — цільовий клас: `1` — манго, `0` — лимон.

Наше завдання — навчити модель **логістичної регресії**, яка безпомилково відрізняє манго від лимона, та проаналізувати якість класифікації.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# 1) Завантаження даних з файлу
df = pd.read_csv('dataset.csv')
df.head()

,sweet,let
0,1,1
1,1,1
2,0,0
3,0,0
4,1,1


### Дослідження
- Імпортуємо бібліотеки:
  - `pandas` — для роботи з таблицями (DataFrame);
  - `LogisticRegression` — модель логістичної регресії для класифікації;
  - `train_test_split` — для поділу даних на тренувальну та тестову вибірки.
- `pd.read_csv('dataset.csv')` — зчитуємо дані з CSV-файлу у датафрейм `df`.
- `df.head()` — показує перші рядки таблиці, щоб переконатися, що дані завантажилися правильно та назви стовпчиків співпадають з умовою завдання (`sweet`, `let`).

In [7]:
# Перевіряємо загальну інформацію про датафрейм
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   sweet   17 non-null     int64
 1   let     17 non-null     int64
dtypes: int64(2)
memory usage: 404.0 bytes


### Дослідження
- `df.info()` показує кількість рядків, назви стовпчиків, типи даних та кількість ненульових значень.
- Переконуємося, що обидва стовпчики `sweet` та `let` мають **числовий тип** (int), а не перетворилися випадково на текст (object).
- Це важливо, бо модель логістичної регресії працює саме з числовими ознаками та мітками класів.

In [8]:
# Перевіряємо, скільки у нас манго (1) і лимонів (0)
print('Розподіл ознаки sweet:')
print(df['sweet'].value_counts(), '\n')
print('Розподіл цільового класу let:')
print(df['let'].value_counts())

Розподіл ознаки sweet:
sweet
1    10
0     7
Name: count, dtype: int64 

Розподіл цільового класу let:
let
1    11
0     6
Name: count, dtype: int64


### Дослідження
- `df['sweet'].value_counts()` показує, скільки разів у наборі даних зустрічається **sweet = 0** та **sweet = 1**.
- `df['let'].value_counts()` показує, скільки у нас **манго (1)** та **лимонів (0)**.
- Так ми розуміємо, чи збалансований наш набір даних і чи вистачає прикладів для кожного класу.
- У нашому випадку обидва класи присутні, тому модель може навчитися їх відрізняти.

In [ ]:
# Вибираємо ознаку (X) та цільову змінну (y)
X = df[['sweet']]  # ознака повинна бути у форматі таблиці (DataFrame)
y = df['let']      # цільовий клас (манго/лимон)

# Ділимо дані на тренувальну та тестову вибірки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Розміри вибірок:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# Створюємо та навчаємо модель логістичної регресії
model = LogisticRegression()
model.fit(X_train, y_train)

# Робимо передбачення на тестових даних
y_pred = model.predict(X_test)

# Обчислюємо точність класифікації
correct = (y_pred == y_test).sum()
total = len(y_test)
accuracy = correct / total

print('Кількість правильних передбачень:', correct)
print('Загальна кількість прикладів у тесті:', total)
print(f'Точність (accuracy) моделі на тестових даних: {accuracy:.1f}')

Розміри вибірок:
X_train: (13, 1)
X_test: (4, 1)
Кількість правильних передбачень: 4
Загальна кількість прикладів у тесті: 4
Точність (accuracy) моделі на тестових даних: 1.0
RMSE: 0.0
R-SQUARED: 1.0


### Дослідження
- Формуємо матрицю ознак `X` — це лише один стовпчик `sweet`, але у форматі таблиці (`[['sweet']]`).
- Цільова змінна `y` — це стовпчик `let`, де **1 = манго**, **0 = лимон**.
- Використовуємо `train_test_split` для поділу даних на:
  - тренувальну вибірку (80% даних) — для навчання моделі;
  - тестову вибірку (20% даних) — для перевірки, як модель працює на нових даних.
- Створюємо об'єкт `LogisticRegression()` та навчаємо модель методом `fit(X_train, y_train)`.
- Після навчання робимо передбачення `y_pred = model.predict(X_test)` для тестових прикладів.
- Точність **accuracy** рахуємо за формулою:

  > точність  =  (кількість правильних передбачень)  /  (загальна кількість передбачень)

- У нашому наборі даних модель класифікує **усі** приклади правильно, тому accuracy = 1.0 (100%).
  Це означає, що модель **безпомилково** відрізняє манго від лимонів на тестовій вибірці.

In [10]:
# Перевіримо, які ймовірності модель дає для лимона (sweet = 0) та манго (sweet = 1)
new_df = pd.DataFrame({'sweet': [0, 1]})
proba = model.predict_proba(new_df)
proba

array([[0.63217985, 0.36782015],
       [0.26245787, 0.73754213]])

### Дослідження
- `model.predict_proba(new_df)` повертає **ймовірності** належності до кожного класу.
- Перший рядок відповідає `sweet = 0` (кислий фрукт), другий — `sweet = 1` (солодкий фрукт).
- У кожному рядку дві колонки:
  - перша — ймовірність класу 0 (лимон),
  - друга — ймовірність класу 1 (манго).
  >Для `sweet = 0` (кислий фрукт):  
   - Ймовірність, що це лимон (0) ≈ 0.63217985 (63.2%)  
   - Ймовірність, що це манго (1) ≈ 0.36782015 (36.8%)  
  >Модель трохи більше схиляється до того, що це лимон, бо 0.632 > 0.367.  
  
  >Для `sweet = 1` (солодкий фрукт):  
   - Ймовірність, що це лимон (0) ≈ 0.26245787 (26.3%)  
   - Ймовірність, що це манго (1) ≈ 0.73754213 (74.7%)  
  >Тут модель вже досить впевнена, що солодкий фрукт — це манго, бо 0.737 значно більше за 0.262.  
- Ми очікуємо, що для `sweet = 0` модель дає вищу ймовірність класу 0 (лимон), а для `sweet = 1` — вищу ймовірність класу 1 (манго).
- Це показує, що модель не просто "видає" класи, а оцінює **ймовірність** того, до якого класу належить фрукт.

## Загальний підсумок
- Побудували модель **логістичної регресії**, яка за однією ознакою `sweet` вміє класифікувати фрукти на **манго (1)** та **лимони (0)**.
- Дані розділили на тренувальну та тестову частини, модель навчена на тренувальній вибірці.
- На тестовій вибірці модель показала **accuracy = 1.0 (100%)**, тобто **жодної помилки** класифікації.
- Ймовірності `predict_proba` показують, що для солодких фруктів (`sweet = 1`) модель впевнено відносить їх до класу манго, а для несолодких (`sweet = 0`) — до класу лимон.

**Висновок:** за рівнем солодкості `sweet` наша модель логістичної регресії **безпомилково відрізняє** соковитий манго від кислого лимона. Це відповідає умові задачі і показує, як працює проста модель класифікації у машинному навчанні.